# Catastrophic Forgetting: clearn vs Baseline

This notebook demonstrates how `clearn` prevents catastrophic forgetting on sequential CIFAR-100 tasks.

**Experiment:**
- Split CIFAR-100 into 20 sequential tasks (5 classes each)
- Train a ResNet-18 on each task in order
- Track Task 1 accuracy as the model learns new tasks
- Compare: Baseline (no protection) vs EWC vs DER++

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rahulkashyap/clearn/blob/main/benchmarks/cifar100_sequential.ipynb)

In [ ]:
# Install clearn (uncomment for Colab)
# !pip install clearn-ai

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import clearn
from clearn.metrics import compute_retention

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load CIFAR-100 and split into 20 tasks (5 classes each)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])

train_dataset = torchvision.datasets.CIFAR100(root="./data", train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR100(root="./data", train=False, download=True, transform=transform)

N_TASKS = 20
CLASSES_PER_TASK = 5

def get_task_loaders(dataset, task_id, batch_size=64):
    """Get a dataloader for a specific task (5 classes)."""
    start_class = task_id * CLASSES_PER_TASK
    end_class = start_class + CLASSES_PER_TASK
    indices = [i for i, (_, label) in enumerate(dataset) if start_class <= label < end_class]
    subset = Subset(dataset, indices)
    return DataLoader(subset, batch_size=batch_size, shuffle=True, num_workers=2)

# Pre-build all loaders
train_loaders = [get_task_loaders(train_dataset, t) for t in range(N_TASKS)]
test_loaders = [get_task_loaders(test_dataset, t) for t in range(N_TASKS)]

print(f"Created {N_TASKS} tasks, {CLASSES_PER_TASK} classes each")
print(f"Task 0 train size: {len(train_loaders[0].dataset)}, test size: {len(test_loaders[0].dataset)}")

In [ ]:
# ResNet-18 adapted for CIFAR-100 (32x32 images)
def make_resnet18():
    model = torchvision.models.resnet18(weights=None, num_classes=100)
    # Adapt for 32x32: smaller initial conv, no maxpool
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model.to(device)

EPOCHS_PER_TASK = 5
LR = 0.01

In [ ]:
# Baseline: standard fine-tuning (no continual learning)
print("Training BASELINE (no protection)...")
baseline_model = make_resnet18()
baseline_task0_acc = []

for task_id in range(N_TASKS):
    optimizer = optim.SGD(baseline_model.parameters(), lr=LR, momentum=0.9)
    loss_fn = nn.CrossEntropyLoss()
    
    baseline_model.train()
    for epoch in range(EPOCHS_PER_TASK):
        for inputs, targets in train_loaders[task_id]:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            loss = loss_fn(baseline_model(inputs), targets)
            loss.backward()
            optimizer.step()
    
    # Evaluate on Task 0
    acc = compute_retention(baseline_model, test_loaders[0], device)
    baseline_task0_acc.append(acc * 100)
    print(f"  Task {task_id:2d} done | Task 0 accuracy: {acc:.1%}")

In [ ]:
# clearn EWC
print("Training with clearn EWC...")
ewc_model = make_resnet18()
ewc_cl = clearn.wrap(ewc_model, strategy="ewc", lambda_=5000)
ewc_task0_acc = []

for task_id in range(N_TASKS):
    optimizer = optim.SGD(ewc_model.parameters(), lr=LR, momentum=0.9)
    ewc_cl.fit(train_loaders[task_id], optimizer, epochs=EPOCHS_PER_TASK, task_id=f"task_{task_id}")
    
    acc = compute_retention(ewc_model, test_loaders[0], device)
    ewc_task0_acc.append(acc * 100)
    print(f"  Task {task_id:2d} done | Task 0 accuracy: {acc:.1%}")

In [ ]:
# clearn DER++
print("Training with clearn DER++...")
der_model = make_resnet18()
der_cl = clearn.wrap(der_model, strategy="der", buffer_size=500, alpha=0.5, beta=1.0)
der_task0_acc = []

for task_id in range(N_TASKS):
    optimizer = optim.SGD(der_model.parameters(), lr=LR, momentum=0.9)
    der_cl.fit(train_loaders[task_id], optimizer, epochs=EPOCHS_PER_TASK, task_id=f"task_{task_id}")
    
    acc = compute_retention(der_model, test_loaders[0], device)
    der_task0_acc.append(acc * 100)
    print(f"  Task {task_id:2d} done | Task 0 accuracy: {acc:.1%}")

In [ ]:
# Plot: Task 0 accuracy across all 20 tasks
plt.figure(figsize=(12, 6))
tasks = list(range(1, N_TASKS + 1))

plt.plot(tasks, baseline_task0_acc, 'r--', linewidth=2, label='Baseline (no CL)', marker='o', markersize=4)
plt.plot(tasks, ewc_task0_acc, 'b-', linewidth=2, label='clearn EWC', marker='s', markersize=4)
plt.plot(tasks, der_task0_acc, 'g-', linewidth=2, label='clearn DER++', marker='^', markersize=4)

plt.xlabel('Tasks Trained', fontsize=14)
plt.ylabel('Task 1 Accuracy (%)', fontsize=14)
plt.title('Catastrophic Forgetting: clearn vs Baseline', fontsize=16, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.ylim(0, 100)
plt.tight_layout()
plt.savefig('forgetting_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFinal Task 1 accuracy after {N_TASKS} tasks:")
print(f"  Baseline: {baseline_task0_acc[-1]:.1f}%")
print(f"  EWC:      {ewc_task0_acc[-1]:.1f}%")
print(f"  DER++:    {der_task0_acc[-1]:.1f}%")

In [ ]:
# Final retention report from DER++ model
print("DER++ Retention Report:")
print(der_cl.diff())

## Key Takeaways

- **Baseline** (standard fine-tuning): Task 1 accuracy collapses as new tasks are learned
- **EWC** (regularization): Protects important weights, maintaining reasonable accuracy
- **DER++** (replay): Best retention by replaying past examples with logit matching

Both `clearn` strategies prevent catastrophic forgetting with just one line of code:

```python
model = clearn.wrap(your_model, strategy="der")  # That's it.
```

Learn more at [clearn.ai](https://clearn.ai)